In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import mahotas as mh
from skimage import io

def extract_hu_moments(image):
    moments = cv2.moments(image)
    hu_moments = cv2.HuMoments(moments).flatten()
    return np.log(1 + abs(hu_moments))  # استفاده از لگاریتم برای بهتر کردن مقادیر

def extract_haralick_features(image):
    try:
        # استخراج ویژگی‌های Haralick با mahotas
        features = mh.features.haralick(image)
        return features.mean(axis=0)  # میانگین برای 4 جهت
    except:
        return np.zeros((13,))  # اگر خطا داد، صفر برگردان

def extract_sift_features(image, max_keypoints=50):
    try:
        sift = cv2.SIFT_create(nfeatures=50)
        keypoints, descriptors = sift.detectAndCompute(image, None)
        if descriptors is None:
            return np.zeros((max_keypoints * 128,))
        descriptors = descriptors[:max_keypoints].flatten()
        padding = max(0, max_keypoints * 128 - len(descriptors))
        return np.pad(descriptors, (0, padding), mode='constant')
    except:
        return np.zeros((max_keypoints * 128,))

def process_image(image_path):
    try:
        # خواندن تصویر
        image = io.imread(image_path)
        if image.ndim == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image.copy()
        
        # تغییر اندازه برای سرعت بیشتر
        gray = cv2.resize(gray, (256, 256))

        # Hu Moments
        hu_features = extract_hu_moments(gray)

        # Haralick Features
        haralick_features = extract_haralick_features(gray)

        # Histogram Equalization
        equalized = cv2.equalizeHist(gray)

        # SIFT Features
        sift_features = extract_sift_features(equalized)

        # ترکیب تمام ویژگی‌ها
        features = np.concatenate((hu_features, haralick_features, sift_features))
        return features
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None

def build_dataset(root_folder):
    data = []
    labels = []

    for label, class_folder in enumerate(os.listdir(root_folder)):
        class_path = os.path.join(root_folder, class_folder)
        if not os.path.isdir(class_path):
            continue
        print(f"Processing class: {class_folder}")
        for image_file in os.listdir(class_path):
            if image_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                image_path = os.path.join(class_path, image_file)
                features = process_image(image_path)
                if features is not None:
                    data.append(features)
                    labels.append(class_folder)

    # تعداد ویژگی‌ها
    num_hu = 7
    num_haralick = 13
    num_sift = 50 * 128

    columns = [f'Hu_{i}' for i in range(num_hu)]
    columns += [f'Haralick_{i}' for i in range(num_haralick)]
    columns += [f'SIFT_{i}' for i in range(num_sift)]
    columns += ['Label']

    df = pd.DataFrame(data, columns=columns[:-1])
    df['Label'] = labels
    return df

# ✅ تنظیمات اصلی
ROOT_FOLDER = "splitted_images/"  # پوشه مرجع شامل زیرپوشهای تصاویر
OUTPUT_CSV = "output_dataset.csv"

df = build_dataset(ROOT_FOLDER)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Dataset saved to {OUTPUT_CSV} with shape {df.shape}")